# 06. Python Lists & Operations (5+ Years Interview Guide)
Deep architectural analysis of dynamic array memory allocation, amortized O(1) appending, Timsort algorithm, slicing pointer mechanics, and copy depth.

### Key 5-Year Interview Concepts Covered:
- **Dynamic Array Representation**: Contiguous pointer arrays in C, over-allocation growth pattern (`0, 4, 8, 16, 25, 35, 46...`).
- **Time Complexities**: Indexing O(1), Append O(1) amortized, Insert/Pop(0) O(N) linear shifts, Search O(N).
- **Slicing Mechanics**: Creating new shallow list copies via pointer copying, step increments, and negative stride reversing.
- **Aliasing & Copy Depth**: Shallow (`list.copy()`, `[:]`) vs deep (`copy.deepcopy()`), list multiplication trap `[[]] * n`.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Dynamic Array Architecture & Mutability
**Explanation**: Python lists (`PyListObject`) are mutable dynamic arrays storing contiguous pointers to objects in heap memory. Because lists store memory references rather than raw contiguous primitive bytes, list items can be heterogeneous (strings, floats, objects). Accessing any element by index is instantaneous O(1) pointer arithmetic.

**Syntax**: `list_var = [elem1, elem2, elem3]` / `list_var = list(iterable)`

In [ ]:
card_providers_list = ['Visa', 'MasterCard']
print(card_providers_list)

### 2. Positive Index Lookups & O(1) Access
**Explanation**: Zero-indexed lookup accesses elements directly via `base_address + (index * pointer_size)`. This guarantees strict O(1) constant time access regardless of list size. Accessing an index outside `[0, len-1]` immediately raises an `IndexError`.

**Syntax**: `item = list_var[index]  # O(1) time complexity`

In [ ]:
card_providers_list = ['Visa', 'MasterCard']
print(card_providers_list[0])

### 3. Negative Index Lookups & Reverse Access
**Explanation**: Negative indexing allows accessing elements relative to the end of the list (`list_var[-1]` accesses the last element). CPython translates `list_var[-k]` internally to `list_var[len(list_var) - k]` in O(1) time without traversing the entire array.

**Syntax**: `last_item = list_var[-1]` / `second_last = list_var[-2]`

In [ ]:
card_providers_list = ['Visa', 'MasterCard']
print(card_providers_list[-1])

### 4. Basic Slicing Boundaries (`[start:stop]`)
**Explanation**: List slicing `list_var[start:stop]` extracts a half-open subset `[start, stop)` from `start` up to but not including `stop`. Slicing creates a brand new list containing copied pointers (a shallow copy) in O(K) time, where K is the slice length. Slices gracefully clamp out-of-bounds indices without raising `IndexError`.

**Syntax**: `sub_list = list_var[start:stop]`

In [ ]:
transaction_amounts_list = [10.0, 20.0, 30.0]
print(transaction_amounts_list[1:3])

### 5. Slice Steps & Strides (`[start:stop:step]`)
**Explanation**: The third slice parameter `step` determines the stride increment. For example, `list_var[::2]` selects every alternate element. Strides are computed in C at native speed and allocate a new list with length `ceil((stop - start) / step)`.

**Syntax**: `stepped_slice = list_var[::step]` / `list_var[1:10:2]`

In [ ]:
transaction_amounts_list = [10.0, 20.0, 30.0, 40.0]
print(transaction_amounts_list[::2])

### 6. Reversing Lists via Slicing (`[::-1]`)
**Explanation**: A negative step `list_var[::-1]` traverses the array backward, returning a reversed shallow copy in O(N) time. While `list_var.reverse()` reverses a list in-place in O(N) time with O(1) auxiliary memory without returning anything, `[::-1]` produces a new list object without altering the original.

**Syntax**: `reversed_list = list_var[::-1]` / `list_var.reverse()  # In-place`

In [ ]:
transaction_amounts_list = [10.0, 20.0, 30.0]
print(transaction_amounts_list[::-1])

### 7. Adding Items: `append()` vs `insert()` vs `extend()`
**Explanation**: `.append(item)` adds an element to the end in amortized O(1) time using CPython's overallocation strategy. `.insert(0, item)` forces every existing element to shift one position to the right, causing a slow O(N) bottleneck (use `collections.deque` for O(1) front prepends). `.extend(iterable)` iterates through the source and appends all items efficiently.

**Syntax**: `list_var.append(x)` / `list_var.extend(iterable)` / `list_var.insert(idx, x)`

In [ ]:
mutable_limits_list = [100.0]
mutable_limits_list.append(200.0)
mutable_limits_list.insert(0, 50.0)
print(mutable_limits_list)

### 8. Concatenation (`+`) vs In-Place Extension (`+=`)
**Explanation**: The `+` operator allocates a new list and copies pointers from both operands, consuming O(N + M) time and extra memory. In contrast, `+=` invokes `__iadd__()`, which extends the existing list in-place (equivalent to `.extend()`) without allocating a new wrapper object.

**Syntax**: `new_list = list_a + list_b` / `list_a += list_b  # In-place`

In [ ]:
mutable_limits_list = [100.0]
mutable_limits_list.extend([200.0, 300.0])
print(mutable_limits_list)

### 9. Deleting Items: `pop()` vs `remove()` vs `del`
**Explanation**: `.pop()` removes and returns the last item in O(1) time; `.pop(0)` removes the first item in O(N) time due to left-shifting. `.remove(value)` scans linearly for the first matching value and removes it in O(N) time (raises `ValueError` if missing). `del list_var[idx]` removes the index and decrements reference count.

**Syntax**: `last = list_var.pop()` / `list_var.remove(value)` / `del list_var[idx]`

In [ ]:
mutable_limits_list = [100.0, 200.0]
mutable_limits_list.remove(100.0)
popped_value = mutable_limits_list.pop()
print(mutable_limits_list, popped_value)

### 10. Sorting with Timsort: `sort()` vs `sorted()`
**Explanation**: Python uses Timsort, an adaptive, stable sorting algorithm combining Merge Sort and Insertion Sort with O(N log N) worst/average time and O(N) best-case time on partially sorted data. `list.sort()` sorts in-place returning `None`, while `sorted(iterable)` accepts any iterable and returns a new sorted list.

**Syntax**: `list_var.sort(key=lambda x: x.amount, reverse=True)` / `sorted(iterable)`

In [ ]:
mutable_limits_list = [300.0, 100.0]
mutable_limits_list.sort()
print(mutable_limits_list)

### 11. Shallow Copy Mechanics
**Explanation**: A shallow copy (`list.copy()`, `list_var[:]`, or `list(list_var)`) creates a new list container, but copies only the memory pointers of the elements. For 1D primitive lists, mutations to the new list do not affect the original. However, for nested lists or mutable objects, both lists point to the identical inner objects.

**Syntax**: `shallow_copy = list_var.copy()` / `shallow_copy = list_var[:]`

In [ ]:
nested_list = [[100.0]]
shallow_copied_list = nested_list.copy()
shallow_copied_list[0].append(200.0)
print('shallow:', shallow_copied_list, '| original:', nested_list)

### 12. Deep Copy Mechanics (`copy.deepcopy`)
**Explanation**: When copying nested data structures (e.g. lists of lists or lists of dicts), `copy.deepcopy()` recursively copies all nested objects, creating completely independent object graphs. It uses an internal memoization dictionary to correctly handle recursive self-referential cycles without infinite recursion.

**Syntax**: `import copy; deep_copy = copy.deepcopy(nested_list)`

In [ ]:
import copy
nested_list = [[100.0]]
deeply_copied_list = copy.deepcopy(nested_list)
deeply_copied_list[0].append(200.0)
print('deeply:', deeply_copied_list, '| original:', nested_list)

### 13. Element Searching & Counting (`index()` & `count()`)
**Explanation**: `.index(val)` performs a linear scan returning the index of the first occurrence in O(N) time (raises `ValueError` if missing). `.count(val)` traverses the full list to count occurrences in O(N) time. In interview questions where frequent lookups or counts are needed, convert the list to a `set` or `collections.Counter` for O(1) lookups.

**Syntax**: `idx = list_var.index(target)` / `cnt = list_var.count(target)`

In [ ]:
amounts_list = [10.0, 20.0, 20.0]
print('Index of 20.0:', amounts_list.index(20.0), 'Count:', amounts_list.count(20.0))

### 14. List Multiplication Aliasing Trap (`[[]] * n`)
**Explanation**: Multiplying a list containing immutables `[0] * 5` is safe (`[0, 0, 0, 0, 0]`). However, multiplying a list containing mutables `[[0]] * 3` creates a list containing 3 references to the EXACT same inner list. Mutating one inner list `grid[0][0] = 99` mutates all rows! Always use list comprehensions for multi-dimensional initialization: `[[0 for _ in range(cols)] for _ in range(rows)]`.

**Syntax**: `grid = [[0] * cols for _ in range(rows)]  # Safe matrix initialization`

In [ ]:
base_list = [1.0]
print(base_list * 3)

### 15. Membership Testing Complexity (List vs Set)
**Explanation**: Checking `item in list_var` performs a sequential linear scan from start to end with O(N) time complexity. For large collections, repeated `in` lookups cause an O(N²) quadratic performance catastrophe. Converting the list to a `set` drops membership checks to O(1) average time via hash table lookups.

**Syntax**: `target in list_var  # O(N) linear time` / `target in set_var  # O(1) hash time`

In [ ]:
allowed_cards_list = ['Visa']
print('Visa' in allowed_cards_list)

### 16. Multi-Dimensional Nested Lists (Matrices)
**Explanation**: Matrices and grids are represented as nested lists `matrix[row][col]`. Traversing row-by-row leverages CPU spatial locality because the outer list's pointers reference contiguous row lists.

**Syntax**: `val = matrix[row_idx][col_idx]`

In [ ]:
nested_matrix_list = [[100.0, 200.0]]
print(nested_matrix_list[0][1])

### 17. List Comprehensions Mechanics & Performance
**Explanation**: List comprehensions `[expr for x in iterable]` run at C-speed using the `LIST_APPEND` bytecode instruction, avoiding the overhead of repeated attribute lookup `.append` calls in Python bytecode loops. They are more readable and faster than standard `for` loops for transforming collections.

**Syntax**: `[x * 2 for x in numbers]`

In [ ]:
squares_payouts_list = [x**2 for x in range(3)]
print(squares_payouts_list)

### 18. List Comprehensions with Filtering Predicates
**Explanation**: Adding an `if` clause `[expr for x in iterable if condition]` filters items during iteration before evaluation. In production, prefer list comprehensions over `list(filter(lambda x: ..., map(lambda x: ..., items)))` because comprehensions avoid extra function call frame overhead.

**Syntax**: `[x for x in numbers if x > threshold]`

In [ ]:
filtered_evens_list = [x for x in range(5) if x%2==0]
print(filtered_evens_list)

### 19. Nested List Comprehensions (Matrix Flattening)
**Explanation**: Nested list comprehensions follow the same ordering as standard nested `for` loops: `[cell for row in matrix for cell in row]`. The outer loop comes first, followed by the inner loop. Keep comprehensions limited to 2 levels to ensure code maintainability.

**Syntax**: `flattened = [item for sublist in matrix for item in sublist]`

In [ ]:
nested_matrix = [[1], [2]]
flattened_list = [y for x in nested_matrix for y in x]
print(flattened_list)

### 20. Type Conversions to List (`list()`)
**Explanation**: The `list(iterable)` constructor consumes any iterable (tuples, sets, dictionary keys, generator objects, range objects) and constructs a new list containing all generated items. Note that for large or infinite generators, calling `list()` consumes memory proportional to the full sequence.

**Syntax**: `items = list(iterable_source)`

In [ ]:
converted_string_list = list('abc')
print(converted_string_list)

## Section 3: Fintech Senior Interview Scenarios
**Explanation**: High-performance list manipulation, deep copy reference isolation, and transaction sequence filtering on real fintech data rows.


In [ ]:
# Solution:
status_list = []
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(20):
        row = f.readline().strip().split(',')
        status_list.append(row[5])
        
completed_count = status_list.count('Completed')
try:
    failed_index = status_list.index('Failed')
except ValueError:
    failed_index = -1
print('Completed counts:', completed_count, '| First Failed Index:', failed_index)


### Q2: Deep Copy Reference Isolation for Transaction Mutex
**Explanation**: **Scenario**: Parse the first 3 transaction records into a nested structure, create an isolated deep copy, mutate a field in the copy, and verify reference isolation to prevent data race conditions.

**Syntax**: `isolated_records = copy.deepcopy(raw_records)`

In [ ]:
# Solution:
import copy
tx_details = []
with open(csv_path, 'r') as f:
    f.readline()
    for _ in range(3):
        row = f.readline().strip().split(',')
        tx_details.append([row[0], row[3]])
        
details_copy = copy.deepcopy(tx_details)
details_copy[0][1] = '0.00'
print('Original:', tx_details[0])
print('Modified Copy:', details_copy[0])
